To run with higer concurrency you'll need to scale up the VM.  By default it runs with 4 cores.  To add a code cell below this that looks like:

```
%%configure -f
{
    "vCores": 16,// Recommended values: [4, 8, 16, 32, 64], Fabric will allocate matched memory according to the specified vCores.
}
```

Note that this will increase the notebook startup time as a VM will need to be spun-up.

## Version 3: several semantic models, user ramp and think times

This notebook runs **one** semantic model at **one** concurrency level. It is orchestrated by two pipelines:

- `PL_PerfTest_MultiModel_v3`: **parallel** ForEach (at most 10) over the list of models.
- `PL_PerfTest_Ramp_v3`: **sequential** ForEach over the user ramp (for example 6, 20, 60, 100, 200, 300) that calls this notebook.

Each model needs its own name (`load_test_name`), its own `dataset` and its own navigation file
(`queryfile`, for example `PowerBIPerformanceData_<model>.json`) in `Files/PerfScenarios/Queries/`.

Think times: with `enable_think_times = True` the pauses between user actions recorded in the JSON file are replayed
(`think_time_scale`, `max_think_time_sec`, `think_time_jitter_pct`). `delay_sec` remains the pause between iterations.
With think times enabled, the per-query wait of `delay_sec` is replaced by the pauses between actions.
`ramp_up_sec` spreads the start of the virtual users and `initial_user_jitter_sec` adds a random initial offset.

RLS impersonation (`EffectiveUserName`) is controlled by `rls_source` (`list`, `file` or `none`) and `rls_scope` (`global` or `per_model`):

| `rls_source` | `rls_scope = global` | `rls_scope = per_model` |
|---|---|---|
| `list` | `rls_users` (comma-separated UPNs) for every model | `users` of the model's item in the `models` parameter; if absent, `rls_users` |
| `file` | `rls_users_file` for every model | `users_file` of the model's item; if absent, `rls_users_file` |
| `none` | no impersonation | no impersonation |

A users file has one UPN per line (blank lines and `#` comments are ignored, duplicates are removed). The UPNs are spread across the
virtual users in turn. With no identities, the identity that runs the test is used.


%%configure -f
{
    "vCores": 4
}

In [ ]:
import uuid
import sempy.fabric
import pandas as pd
import polars as pl
import os
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
import sempy
import time


In [ ]:
# Run summary helpers (standard library only; unit-tested outside Fabric)
# One summary per run (one model at one concurrency level), built from the CSV files of the virtual users.
import csv as _csv
import json
import math
import os
import sys as _sys
from datetime import datetime, timezone

_csv.field_size_limit(min(_sys.maxsize, 2**31 - 1))  # the DAX text of a query can be long


def _percentile(values, pct):
    """Linear interpolation, same as numpy's default and DuckDB's quantile_cont."""
    if not values:
        return None
    data = sorted(values)
    k = (len(data) - 1) * pct / 100.0
    lo = math.floor(k)
    hi = min(lo + 1, len(data) - 1)
    return data[lo] + (data[hi] - data[lo]) * (k - lo)


def _iso_utc(epoch):
    return None if epoch is None else datetime.fromtimestamp(epoch, tz=timezone.utc).isoformat().replace("+00:00", "Z")


def read_run_rows(folder_path):
    """Rows of every <loadtestId>_user_<k>.csv of a run folder (the thinktimes/ sub-folder is not read)."""
    rows = []
    if not os.path.isdir(folder_path):
        return rows
    for name in sorted(os.listdir(folder_path)):
        if name.endswith(".csv") and "_user_" in name:
            with open(os.path.join(folder_path, name), newline="", encoding="utf-8") as handle:
                rows.extend(_csv.DictReader(handle))
    return rows


def summarize_run_folder(folder_path, loadtest_id, load_test_name, model, concurrent_threads, run_error=None):
    """Latency percentiles, failed queries and lost virtual users of one run.

    A row with status 'error' is a query that failed; its duration is not used for the percentiles. A virtual user
    without a CSV file failed before running any query: its queries count as failed too (expected = users x queries
    per user).
    """
    rows = read_run_rows(folder_path)
    concurrent_threads = int(concurrent_threads)
    per_user = {}
    for r in rows:
        per_user[r.get("thread_id")] = per_user.get(r.get("thread_id"), 0) + 1
    queries_per_user = max(per_user.values()) if per_user else 0
    ok = [float(r["duration"]) for r in rows if (r.get("status") or "ok") == "ok"]
    failed = sum(1 for r in rows if (r.get("status") or "ok") != "ok")
    missing_users = max(0, concurrent_threads - len(per_user))
    expected = concurrent_threads * queries_per_user
    lost = missing_users * queries_per_user
    starts = [float(r["start_time"]) for r in rows]
    ends = [float(r["start_time"]) + float(r["duration"]) for r in rows]
    wall = (max(ends) - min(starts)) if rows else 0.0
    error_log = os.path.join(folder_path, "error.log")
    error_lines = 0
    if os.path.exists(error_log):
        with open(error_log, encoding="utf-8", errors="replace") as handle:
            error_lines = sum(1 for line in handle if line.strip())
    if run_error or not rows:
        status = "failed"
    elif failed or missing_users:
        status = "completed_with_errors"
    else:
        status = "completed"

    def r4(x):
        return None if x is None else round(x, 4)

    return {
        "loadtest_id": loadtest_id,
        "load_test_name": load_test_name,
        "model": model,
        "concurrent_threads": concurrent_threads,
        "status": status,
        "users_with_results": len(per_user),
        "missing_users": missing_users,
        "queries_per_user": queries_per_user,
        "queries": len(rows),
        "failed_queries": failed,
        "expected_queries": expected,
        "error_rate": round((failed + lost) / expected, 4) if expected else (1.0 if status == "failed" else 0.0),
        "mean_s": r4(sum(ok) / len(ok)) if ok else None,
        "p50_s": r4(_percentile(ok, 50)),
        "p90_s": r4(_percentile(ok, 90)),
        "p95_s": r4(_percentile(ok, 95)),
        "p99_s": r4(_percentile(ok, 99)),
        "max_s": r4(max(ok)) if ok else None,
        "qps": round(len(rows) / wall, 3) if wall > 0 else None,
        "start_utc": _iso_utc(min(starts)) if rows else None,
        "end_utc": _iso_utc(max(ends)) if rows else None,
        "duration_s": round(wall, 1),
        "error_log_lines": error_lines,
        "run_error": (str(run_error)[:1000] if run_error else None),
    }


def write_run_summary(summary, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    with open(os.path.join(folder_path, "summary.json"), "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=1)


# Delta schemas: explicit, so that columns that are empty in a run (for example run_error) never break type inference
RUN_TABLE = "loadtest_runs"
QUERY_TABLE = "loadtest_queries"
RUN_TABLE_SCHEMA = [
    ("loadtest_id", "string"), ("load_test_name", "string"), ("model", "string"), ("concurrent_threads", "bigint"),
    ("status", "string"), ("users_with_results", "bigint"), ("missing_users", "bigint"), ("queries_per_user", "bigint"),
    ("queries", "bigint"), ("failed_queries", "bigint"), ("expected_queries", "bigint"), ("error_rate", "double"),
    ("mean_s", "double"), ("p50_s", "double"), ("p90_s", "double"), ("p95_s", "double"), ("p99_s", "double"),
    ("max_s", "double"), ("qps", "double"), ("start_utc", "string"), ("end_utc", "string"), ("duration_s", "double"),
    ("error_log_lines", "bigint"), ("run_error", "string"),
]
QUERY_TABLE_SCHEMA = [
    ("loadtest_id", "string"), ("model", "string"), ("concurrent_threads", "bigint"), ("thread_id", "bigint"),
    ("effective_username", "string"), ("iteration", "bigint"), ("query_number", "bigint"), ("action_index", "bigint"),
    ("source_label", "string"), ("visual_name", "string"), ("visual_id", "string"), ("query", "string"),
    ("rows", "bigint"), ("duration", "double"), ("start_time", "double"), ("applied_think_time_sec", "double"),
    ("status", "string"), ("error", "string"),
]


def _typed(value, kind):
    if value is None or value == "":
        return None
    if kind == "bigint":
        return int(float(value))
    if kind == "double":
        return float(value)
    return str(value)


def table_rows(records, schema):
    """Tuples in schema order, with the declared types; missing columns (older CSV files) become NULL."""
    return [tuple(_typed(rec.get(name), kind) for name, kind in schema) for rec in records]


_PANDAS_DTYPES = {"bigint": "Int64", "double": "float64", "string": "string"}


def delta_frame(records, schema):
    """pandas DataFrame with the declared types. Nullable Int64 and string dtypes, so an all-null column (run_error)
    still has an Arrow type."""
    import pandas as pd
    frame = pd.DataFrame(table_rows(records, schema), columns=[name for name, _ in schema])
    return frame.astype({name: _PANDAS_DTYPES[kind] for name, kind in schema})


def append_to_delta(write, uri, frame, storage_options):
    """Append with deltalake's write_deltalake. RunLoadTest_v3 is a Python notebook: there is no Spark session.
    schema_mode="merge" adds columns a newer version brings; deltalake versions without it (or with the pyarrow engine)
    fall back to a plain append, which needs the table schema to match (drop the table after a schema change)."""
    last_error = None
    for extra in ({"schema_mode": "merge"}, {"schema_mode": "merge", "engine": "rust"}, {}):
        try:
            return write(uri, frame, mode="append", storage_options=storage_options, **extra)
        except (TypeError, ValueError) as error:  # option not supported by the installed deltalake
            last_error = error
    raise last_error


### Parameters
The next cell is tagged **Parameters** and holds the defaults for a manual run.
When the pipeline calls this notebook, Fabric inserts an *injected parameters* cell right below it that overrides
`load_test_name`, `dataset`, `workspace`, `queryfile`, `concurrent_threads`, `delay_sec`, `iterations`, the RLS settings,
`stop_on_query_error`, `fail_on_error_rate`, `write_results_to_tables` and the think-time and ramp parameters. Think times are **disabled** by default (original behaviour).

At the end the notebook writes `summary.json` in the run folder (latency percentiles, failed queries, lost virtual users)
and exits with that summary as its **exit value**, so a pipeline can read it from
`activity('<notebook activity>').output.result.exitValue`. With `write_results_to_tables = True` the run is also appended to
the Delta tables `loadtest_runs` (one row per run) and `loadtest_queries` (one row per query) of the default lakehouse.

A failed query is recorded as a row with `status = error` and the virtual user goes on; with `stop_on_query_error = True` the
user stops at its first failed query instead (behaviour before 3.4.0). In both cases its CSV is written.
With the default, **a run with failed queries still ends `Completed`**: read `failed_queries` in `summary.json` or in the
exit value, or set `fail_on_error_rate` (for example `0.05`) to make the notebook fail above that error rate.

In [ ]:
load_test_name = "My test"                                                         #name of test for logging
dataset = "SEMANTIC_MODEL_NAME"                                                     #name of the semantic model to test
workspace = None                                                                    #workspace containing the semantic model to test "None" for the current workspace

queryfile = "/lakehouse/default/Files/PerfScenarios/Queries/PowerBIPerformanceData.json"    #Power BI Desktop performance analyzer file containing the use case to test
concurrent_threads = 6                                                                      #number of concurrent threads to test
delay_sec = 4                                                                               #number of seconds each user waits between iterations of the use case
iterations = 1                                                                              #number of iterations run by each user

# --- Think times (disabled by default: same behaviour as the original notebook) ---
enable_think_times = False                                                                  #replay the pauses between user actions recorded in the JSON file
think_time_scale = 1.0                                                                      #1.0 = recorded pace; 0.5 = twice as fast; 2.0 = twice as slow; 0.0 = no internal pauses
max_think_time_sec = 60.0                                                                   #cap applied to any single pause
think_time_jitter_pct = 0.0                                                                 #0.10 = +/-10 % random variation on each pause
fallback_think_time_sec = 0.0                                                               #pause between queries when the JSON file has no User Action events

# --- Staggered start (ramp inside each level) ---
ramp_up_sec = 0.0                                                                           #window over which the virtual users start; 0 = all at once
initial_user_jitter_sec = 0.0                                                               #extra random offset (0..N s) before the first action

# --- RLS identities (EffectiveUserName) ---
rls_source = "list"                                                                         #where the identities come from: "list" (rls_users / models[].users), "file" (text file, one UPN per line) or "none"
rls_scope = "global"                                                                        #"global" = the same identities for every model; "per_model" = each model uses its own (models[].users or models[].users_file)
rls_users = ""                                                                              #global list: comma-separated UPNs spread across the virtual users; empty = no impersonation
rls_users_file = ""                                                                         #global file, for example /lakehouse/default/Files/PerfScenarios/Users/users.txt (one UPN per line)
model_settings = ""                                                                         #JSON of the models[] item of this model (filled in by the pipeline); keys used here: users, users_file

# --- Results ---
stop_on_query_error = False                                                                 #True = a virtual user stops at its first failed query (behaviour before 3.4.0)
fail_on_error_rate = 1.0                                                                    #the notebook fails when the run's error_rate is above this value (0.05 = 5 %); 1.0 = never
write_results_to_tables = False                                                             #also append the run to the Delta tables loadtest_runs and loadtest_queries of the default lakehouse


In [ ]:
# Values received from the pipeline ('injected parameters' cell) or, in a manual run, those of the Parameters cell
import json
import re


def _as_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


concurrent_threads = int(concurrent_threads)
delay_sec = int(delay_sec) if float(delay_sec).is_integer() else float(delay_sec)
iterations = int(iterations)
enable_think_times = _as_bool(enable_think_times)
write_results_to_tables = _as_bool(write_results_to_tables)
stop_on_query_error = _as_bool(stop_on_query_error)
fail_on_error_rate = float(fail_on_error_rate)
think_time_scale = float(think_time_scale)
max_think_time_sec = float(max_think_time_sec)
think_time_jitter_pct = float(think_time_jitter_pct)
fallback_think_time_sec = float(fallback_think_time_sec)
ramp_up_sec = float(ramp_up_sec)
initial_user_jitter_sec = float(initial_user_jitter_sec)
dataset = (dataset or "").strip()
rls_source = str(rls_source or "list").strip().lower()
rls_scope = str(rls_scope or "global").strip().lower()
rls_users_file = str(rls_users_file or "").strip()
if rls_source not in ("list", "file", "none"):
    raise ValueError(f"rls_source must be 'list', 'file' or 'none': {rls_source!r}")
if rls_scope not in ("global", "per_model"):
    raise ValueError(f"rls_scope must be 'global' or 'per_model': {rls_scope!r}")
try:
    model_settings_doc = json.loads(model_settings) if str(model_settings or "").strip() else {}
except ValueError as exc:
    raise ValueError(f"model_settings is not valid JSON: {exc}") from exc
if not isinstance(model_settings_doc, dict):
    raise ValueError("model_settings must be a JSON object (the item of the models parameter)")


def _parse_upns(value, origin):
    """UPNs from a list or from text (one per line, or separated by commas or semicolons). Comments start with '#'."""
    if isinstance(value, (list, tuple)):
        value = "\n".join(str(v) for v in value)
    found, invalid = {}, []
    for number, line in enumerate(str(value or "").splitlines(), start=1):
        line = re.sub(r"(^|\s)#.*$", "", line)
        for token in re.split(r"[,;]", line):
            token = token.strip()
            if not token:
                continue
            if re.fullmatch(r"[^@\s]+@[^@\s]+", token):
                found.setdefault(token.lower(), token)  # duplicates are removed, the first spelling is kept
            else:
                invalid.append(f"{origin}, line {number}: {token!r}")
    if invalid:
        raise ValueError("Not a valid user principal name (expected user@domain):\n  " + "\n  ".join(invalid[:10]))
    return list(found.values())


def _read_users_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Users file not found: {path}")
    with open(path, encoding="utf-8-sig") as f:
        upns = _parse_upns(f.read(), path)
    if not upns:
        raise ValueError(f"Users file contains no users: {path}")
    return upns


def resolve_rls_users():
    """Returns (UPNs, description of where they come from) according to rls_source and rls_scope."""
    if rls_source == "none":
        return [], "none (rls_source = none)"
    per_model = rls_scope == "per_model"
    if rls_source == "list":
        own = _parse_upns(model_settings_doc.get("users"), "models[].users") if per_model else []
        if own:
            return own, "list of this model (models[].users)"
        upns = _parse_upns(rls_users, "rls_users")
        return upns, "global list (rls_users)" if upns else "none (no users in the list)"
    own_file = str(model_settings_doc.get("users_file") or "").strip() if per_model else ""
    if own_file:
        return _read_users_file(own_file), f"file of this model (models[].users_file): {own_file}"
    if not rls_users_file:
        raise ValueError("rls_source = 'file' but no file was given (set rls_users_file or, with rls_scope = 'per_model', users_file in the model)")
    return _read_users_file(rls_users_file), f"global file (rls_users_file): {rls_users_file}"


rls_user_list, rls_origin = resolve_rls_users()
workspace = workspace.strip() if isinstance(workspace, str) and workspace.strip() else None

if not dataset:
    raise ValueError("dataset (semantic model) is required")
if not 1 <= concurrent_threads <= 300:
    raise ValueError(f"concurrent_threads out of range (1-300): {concurrent_threads}")
if not 0.0 <= fail_on_error_rate <= 1.0:
    raise ValueError("fail_on_error_rate must be between 0.0 and 1.0")
if think_time_scale < 0:
    raise ValueError("think_time_scale cannot be negative")
if max_think_time_sec < 0:
    raise ValueError("max_think_time_sec cannot be negative")
if not 0.0 <= think_time_jitter_pct <= 1.0:
    raise ValueError("think_time_jitter_pct must be between 0.0 and 1.0")
if min(ramp_up_sec, initial_user_jitter_sec, fallback_think_time_sec) < 0:
    raise ValueError("ramp_up_sec, initial_user_jitter_sec and fallback_think_time_sec cannot be negative")
if not os.path.exists(queryfile):
    raise FileNotFoundError(f"Navigation file for model '{dataset}' not found: {queryfile}")
print(f"RLS identities: {len(rls_user_list) if rls_user_list else 'none (the identity running the test is used)'}"
      f" | source: {rls_origin} | rls_source={rls_source}, rls_scope={rls_scope}")
if rls_user_list and concurrent_threads > len(rls_user_list):
    print(f"Note: {concurrent_threads} virtual users share {len(rls_user_list)} identities; they are reused in turn.")
print(f"Active parameters: model={dataset!r}, load_test_name={load_test_name!r}, concurrent_threads={concurrent_threads}, "
      f"delay_sec={delay_sec}, iterations={iterations}, think_times={enable_think_times} (scale={think_time_scale}, "
      f"max={max_think_time_sec}s, jitter={think_time_jitter_pct:.0%}), ramp_up_sec={ramp_up_sec}, queryfile={queryfile}")

workspace = workspace if workspace else notebookutils.runtime.context['currentWorkspaceName']

if 'Collapse Here':
    ts=time.strftime("%Y%m%d-%H%M%S")
    loadtestId:str = f"{load_test_name}-c{concurrent_threads}-{ts}"
    print(loadtestId)

    lakehouseId = sempy.fabric.get_lakehouse_id()
    workspaceId = sempy.fabric.get_workspace_id()

    # Path for storing CSV files during run
    folder_path = f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}"
    # Used for writing combined logs


    #Create folder dedicated to logs from this run only
    notebookutils.fs.mkdirs(folder_path)


    users = rls_user_list  # resolved from rls_source / rls_scope; empty = no EffectiveUserName
                    # 
                    # users = ["user1@domain.com","user@domain.com","user3@domain.com"]
                

    args = { 
        "xmla_endpoint" : None, #defaults to current workspace's endpoint
        "workspace" : workspace,
        "perf_analyzer_filename" : queryfile , 
        "model" : dataset,
        "roles" : None,
        "customdata" : None,
        "effective_username" : None,
        "iterations" : iterations,
        "delay_sec" : delay_sec,
        "loadtestId" : loadtestId ,
        "threadId" : 0 ,
        "concurrent_threads" : concurrent_threads,
        "useRootDefaultLakehouse": True,
        "load_test_name" : load_test_name,
        "enable_think_times" : enable_think_times,
        "think_time_scale" : think_time_scale,
        "max_think_time_sec" : max_think_time_sec,
        "think_time_jitter_pct" : think_time_jitter_pct,
        "fallback_think_time_sec" : fallback_think_time_sec,
        "initial_user_jitter_sec" : initial_user_jitter_sec,
        "start_delay_sec" : 0.0,
        "stop_on_query_error" : stop_on_query_error
    }

    activity =  {
                "name": "RunPerfScenario", # activity name, must be unique
                "path": "RunPerfScenario_v3", # notebook path
                "timeoutPerCellInSeconds": 9000, # max timeout for each cell, default to 90 seconds
                "args": {"param1": "value1"}, # notebook parameters
                "workspace": None, # workspace name, default to current workspace
                "retry": 0, # max retry times, default to 0
                "retryIntervalInSeconds": 0, # retry interval, default to 0 seconds
                "dependencies": [] # list of activity names that this activity depends on
            }


    DAG = {
        "activities": [],
        "timeoutInSeconds": 43200, # max timeout for the entire pipeline, default to 12 hours
        "concurrency": 25 # max number of notebooks to run concurrently, default to 25
    }
    DAG["concurrency"] = concurrent_threads

    for i in range(concurrent_threads):
        a = activity.copy()
        a["name"] = f"{a['name']}_{i}"
        this_args = args.copy()
        if (len(users)>0):
            user = users[i%len(users)]
            this_args["effective_username"] = user 
            #this_args["customdata"] = user

        this_args["threadId"] = i
        this_args["start_delay_sec"] = ramp_up_sec * i / concurrent_threads  # staggered start
        a["args"] = this_args

        DAG["activities"].append(a)
        
    # print(DAG)
    print('running load test')
    try:

        ####################################
        ##  Run the parallel threads here ##
        ####################################

        results = notebookutils.notebook.runMultiple(DAG) 


    except Exception as e:
        # keep the evidence of a failed run: summary.json with the error and what the surviving users measured
        write_run_summary(summarize_run_folder(folder_path, loadtestId, load_test_name, dataset, concurrent_threads,
                                               run_error=f"{type(e).__name__}: {e}"), folder_path)
        raise RuntimeError(f"Load test {loadtestId} failed") from e
    print('load test complete')

    # Consolodate CSV log files of results into single dataframe
    df_list = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            polars_df = pl.read_csv(file_path)
            # Need to store Time using format that will work for Direct Lake

            polars_df = (polars_df
                .with_columns([
                    pl.col("start_time_dt").str.to_datetime(time_unit="ms").dt.truncate("1s").alias("start_time_s").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            polars_df = (polars_df
                .with_columns([
                    pl.col("start_time_dt").str.to_datetime(time_unit="ms").alias("start_time_dt").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            df_list.append(polars_df)

    try:
        # files can differ in columns (for example older runs without status/error) or in inferred types
        combined_df = pl.concat(df_list, how="diagonal_relaxed")
    except Exception:
        combined_df = pl.concat(df_list)

In [ ]:
combined_df = combined_df.with_columns(pl.col("start_time_s").cast(pl.Datetime).alias("start_time_s_dt"))


combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").mean().alias("avg1"))
      .join(combined_df, on="start_time_s_dt")
      .select("avg1")
)

combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").quantile(0.9).alias("p90"))
      .join(combined_df, on="start_time_s_dt")
      .select("p90")
)

combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").quantile(0.5).alias("p50"))
      .join(combined_df, on="start_time_s_dt")
      .select("p50")
)

display(combined_df)


In [ ]:
# Run summary: summary.json next to the CSV files, optional Delta tables and the exit value of this notebook.
# A pipeline reads the exit value as activity('<activity>').output.result.exitValue (a JSON string).
run_summary = summarize_run_folder(folder_path, loadtestId, load_test_name, dataset, concurrent_threads)
write_run_summary(run_summary, folder_path)
print(json.dumps(run_summary, indent=1))

if write_results_to_tables:
    # Python notebook (no Spark): deltalake over the OneLake abfss path; the mounted /lakehouse/default/Tables path does not
    # support the rename that a Delta commit needs. Lakehouses with schemas keep the tables in dbo.
    ctx = notebookutils.runtime.context
    tables_root = f"abfss://{ctx['currentWorkspaceId']}@onelake.dfs.fabric.microsoft.com/{ctx['defaultLakehouseId']}/Tables"
    if os.path.isdir("/lakehouse/default/Tables/dbo"):
        tables_root += "/dbo"
    delta_options = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
    append_to_delta(write_deltalake, f"{tables_root}/{RUN_TABLE}", delta_frame([run_summary], RUN_TABLE_SCHEMA), delta_options)
    append_to_delta(write_deltalake, f"{tables_root}/{QUERY_TABLE}",
                    delta_frame(read_run_rows(folder_path), QUERY_TABLE_SCHEMA), delta_options)
    print(f"Appended to the Delta tables {RUN_TABLE} and {QUERY_TABLE}")

# A run with failed queries ends Completed by default: read failed_queries in summary.json or the exit value.
if (run_summary.get("error_rate") or 0) > fail_on_error_rate:
    raise RuntimeError(f"Load test {loadtestId}: error rate {run_summary['error_rate']:.1%} is above fail_on_error_rate "
                       f"({fail_on_error_rate:.1%}); see summary.json and error.log")

notebookutils.notebook.exit(json.dumps(run_summary))